In [1]:
import requests
import time




In [2]:
GITHUB_HEADERS = {
    "Accept":     "application/vnd.github+json",
    "User-Agent": "ZHAW-BigData-Explorer/1.0",
}

last_etag: str  = ""

GITHUB_API_URL = "https://api.github.com/events"
etag_headers = {"If-None-Match": last_etag} if last_etag else {}

page = 0

resp = requests.get(
    GITHUB_API_URL,
    headers={**GITHUB_HEADERS, **etag_headers},
    params={"per_page": 30, "page": page},
    timeout=10,
)


In [3]:
resp


<Response [200]>

In [4]:
print(resp.status_code)
print(resp.headers)
print(resp.text)
print(resp.json())


200
{'Date': 'Wed, 01 Apr 2026 12:38:51 GMT', 'Content-Type': 'application/json; charset=utf-8', 'Cache-Control': 'public, max-age=300, s-maxage=300', 'Vary': 'Accept,Accept-Encoding, Accept, X-Requested-With', 'ETag': 'W/"a166ab2b30bb4c1306c90e71f7f0eab26e957ddabd19bec1ed0ce017829e1837"', 'Last-Modified': 'Wed, 01 Apr 2026 12:33:50 GMT', 'X-Poll-Interval': '60', 'X-GitHub-Media-Type': 'github.v3; format=json', 'Link': '<https://api.github.com/events?per_page=30&page=2>; rel="next", <https://api.github.com/events?per_page=30&page=10>; rel="last"', 'x-github-api-version-selected': '2022-11-28', 'Access-Control-Expose-Headers': 'ETag, Link, Location, Retry-After, X-GitHub-OTP, X-RateLimit-Limit, X-RateLimit-Remaining, X-RateLimit-Used, X-RateLimit-Resource, X-RateLimit-Reset, X-OAuth-Scopes, X-Accepted-OAuth-Scopes, X-Poll-Interval, X-GitHub-Media-Type, X-GitHub-SSO, X-GitHub-Request-Id, Deprecation, Sunset', 'Access-Control-Allow-Origin': '*', 'Strict-Transport-Security': 'max-age=31536

In [5]:
# Basis-Setup: Imports und Anzeigeoptionen
import os
import pandas as pd
from sqlalchemy import create_engine, text
import psycopg2
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


# 1) Build connection (Docker-Compose compatible)
# Priority: full URL from ENV -> otherwise build from individual values

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")        # docker-compose maps 5432:5432
DB_NAME = os.getenv("DB_NAME", "github_events")
DB_USER = os.getenv("DB_USER", "github")
DB_PASSWORD = os.getenv("DB_PASSWORD", "github_secret")

DATABASE_URL = os.getenv("DATABASE_URL") or (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)
print(f"DB connection ready: {DB_HOST}:{DB_PORT}/{DB_NAME}")

DB connection ready: localhost:5432/github_events


In [ ]:
source_table = "events"
query = f"SELECT * FROM {source_table}"
with engine.connect() as conn:
    repo_df = pd.read_sql(text(query), conn)

print(f"Quelle: {source_table} | Geladene Zeilen: {len(repo_df):,}")
repo_df